In [1]:
import pandas as pd
import duckdb

In [2]:
df = pd.read_csv(
    'Cleaned_Product_Demand.csv',
    parse_dates=['Date']
)

df.head()

,Product_Code,Warehouse,Product_Category,Date,Order_Demand,Demand_Type
0,Product_0993,Whse_J,Category_028,2012-07-27,100,Positive
1,Product_0979,Whse_J,Category_028,2012-01-19,500,Positive
2,Product_0979,Whse_J,Category_028,2012-02-03,500,Positive
3,Product_0979,Whse_J,Category_028,2012-02-09,500,Positive
4,Product_0979,Whse_J,Category_028,2012-03-02,500,Positive


In [3]:
con = duckdb.connect()

In [4]:
con.execute("""
    CREATE TABLE product_demand AS
    SELECT * FROM df
""")

In [5]:
con.execute("""
    SELECT *
    FROM product_demand
    LIMIT 5
""").df()

,Product_Code,Warehouse,Product_Category,Date,Order_Demand,Demand_Type
0,Product_0993,Whse_J,Category_028,2012-07-27,100,Positive
1,Product_0979,Whse_J,Category_028,2012-01-19,500,Positive
2,Product_0979,Whse_J,Category_028,2012-02-03,500,Positive
3,Product_0979,Whse_J,Category_028,2012-02-09,500,Positive
4,Product_0979,Whse_J,Category_028,2012-03-02,500,Positive


In [6]:
con.execute("""
    SELECT COUNT(*) AS Total_Records
    FROM product_demand
""").df()

,Total_Records
0,1036643


In [7]:
con.execute("""
    DESCRIBE product_demand
""").df()

,column_name,column_type,null,key,default,extra
0,Product_Code,VARCHAR,YES,None,None,None
1,Warehouse,VARCHAR,YES,None,None,None
2,Product_Category,VARCHAR,YES,None,None,None
3,Date,TIMESTAMP_NS,YES,None,None,None
4,Order_Demand,BIGINT,YES,None,None,None
5,Demand_Type,VARCHAR,YES,None,None,None


# Phase 4: SQL Business Analysis

## Q1. What are the key demand and supply chain KPIs?

In [8]:
con.execute("""
    SELECT
        COUNT(*) AS Total_Records,
        COUNT(DISTINCT Product_Code) AS Total_Products,
        COUNT(DISTINCT Warehouse) AS Total_Warehouses,
        COUNT(DISTINCT Product_Category) AS Total_Categories,
        SUM(Order_Demand) AS Net_Demand,
        SUM(CASE WHEN Order_Demand > 0
            THEN Order_Demand ELSE 0 END) AS Positive_Demand,
        ABS(SUM(CASE WHEN Order_Demand < 0
            THEN Order_Demand ELSE 0 END)) AS Negative_Demand,
        ROUND(AVG(Order_Demand), 2) AS Average_Demand,
        MEDIAN(Order_Demand) AS Median_Demand
    FROM product_demand
""").df()

,Total_Records,Total_Products,Total_Warehouses,Total_Categories,Net_Demand,Positive_Demand,Negative_Demand,Average_Demand,Median_Demand
0,1036643,2160,4,33,5.096047e+09,5.110781e+09,14733896.0,4915.91,300.0


## Q2. How concentrated is demand among the top 10, 20 and 50 products?

In [9]:
con.execute("""
    WITH product_totals AS (
        SELECT
            Product_Code,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        GROUP BY Product_Code
    ),
    ranked_products AS (
        SELECT *,
            ROW_NUMBER() OVER (
                ORDER BY Total_Demand DESC, Product_Code
            ) AS Product_Rank
        FROM product_totals
    )
    SELECT
        SUM(Total_Demand) AS Overall_Demand,

        SUM(CASE WHEN Product_Rank <= 10
            THEN Total_Demand ELSE 0 END) AS Top_10_Demand,

        ROUND(
            SUM(CASE WHEN Product_Rank <= 10
                THEN Total_Demand ELSE 0 END)
            * 100.0 / SUM(Total_Demand), 2
        ) AS Top_10_Share,

        ROUND(
            SUM(CASE WHEN Product_Rank <= 20
                THEN Total_Demand ELSE 0 END)
            * 100.0 / SUM(Total_Demand), 2
        ) AS Top_20_Share,

        ROUND(
            SUM(CASE WHEN Product_Rank <= 50
                THEN Total_Demand ELSE 0 END)
            * 100.0 / SUM(Total_Demand), 2
        ) AS Top_50_Share

    FROM ranked_products
""").df()

,Overall_Demand,Top_10_Demand,Top_10_Share,Top_20_Share,Top_50_Share
0,5.096047e+09,1.766641e+09,34.67,50.38,70.91


## Q3. Which are the top five products within each warehouse?

In [10]:
con.execute("""
    WITH warehouse_products AS (
        SELECT
            Warehouse,
            Product_Code,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        GROUP BY Warehouse, Product_Code
    ),
    ranked_products AS (
        SELECT *,
            DENSE_RANK() OVER (
                PARTITION BY Warehouse
                ORDER BY Total_Demand DESC
            ) AS Product_Rank
        FROM warehouse_products
    )
    SELECT *
    FROM ranked_products
    WHERE Product_Rank <= 5
    ORDER BY Warehouse, Product_Rank
""").df()

,Warehouse,Product_Code,Total_Demand,Product_Rank
0,Whse_A,Product_1496,10265950.0,1
1,Whse_A,Product_1470,9301400.0,2
2,Whse_A,Product_1388,7649100.0,3
3,Whse_A,Product_0424,6180000.0,4
4,Whse_A,Product_1513,5918200.0,5
5,Whse_C,Product_1245,88878000.0,1
6,Whse_C,Product_1480,60567000.0,2
7,Whse_C,Product_1341,41536000.0,3
8,Whse_C,Product_1278,36146000.0,4
9,Whse_C,Product_1502,26222500.0,5


## Q4. Which high-demand products have demand concentrated in a single warehouse?

In [11]:
con.execute("""
    SELECT
        Product_Code,
        Product_Category,
        COUNT(DISTINCT Warehouse) AS Warehouse_Count,
        MAX(Warehouse) AS Warehouse,
        SUM(Order_Demand) AS Total_Demand,
        COUNT(*) AS Total_Records
    FROM product_demand
    GROUP BY Product_Code, Product_Category
    HAVING COUNT(DISTINCT Warehouse) = 1
    ORDER BY Total_Demand DESC
    LIMIT 10
""").df()

,Product_Code,Product_Category,Warehouse_Count,Warehouse,Total_Demand,Total_Records
0,Product_1359,Category_019,1,Whse_J,470610000.0,16935
1,Product_1248,Category_019,1,Whse_J,288927000.0,6348
2,Product_1295,Category_019,1,Whse_J,121992000.0,10575
3,Product_1286,Category_019,1,Whse_J,101331800.0,8888
4,Product_1274,Category_019,1,Whse_J,92755000.0,7047
5,Product_1451,Category_019,1,Whse_J,91799000.0,4702
6,Product_1287,Category_019,1,Whse_J,88484000.0,7829
7,Product_1378,Category_019,1,Whse_J,83325300.0,9770
8,Product_1382,Category_019,1,Whse_J,81554200.0,8619
9,Product_1294,Category_019,1,Whse_J,78067600.0,8384


In [12]:
con.execute("""
    WITH single_warehouse_products AS (
        SELECT
            Product_Code,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        GROUP BY Product_Code
        HAVING COUNT(DISTINCT Warehouse) = 1
        ORDER BY Total_Demand DESC
        LIMIT 10
    )
    SELECT
        SUM(Total_Demand) AS Combined_Demand,
        ROUND(
            SUM(Total_Demand) * 100.0 /
            (SELECT SUM(Order_Demand) FROM product_demand),
            2
        ) AS Overall_Demand_Share,
        ROUND(
            SUM(Total_Demand) * 100.0 /
            (SELECT SUM(Order_Demand)
             FROM product_demand
             WHERE Warehouse = 'Whse_J'),
            2
        ) AS Warehouse_Demand_Share
    FROM single_warehouse_products
""").df()

,Combined_Demand,Overall_Demand_Share,Warehouse_Demand_Share
0,1.498846e+09,29.41,44.82


### Insights

- All ten of the highest-demand products with demand recorded in a single warehouse belong to Whse_J and Category_019.
- Product_1359 records the highest net demand of 470.61 million units, followed by Product_1248 at 288.93 million units.
- The results indicate significant demand concentration across a particular warehouse and product category.

### Business Recommendation

Prioritise demand monitoring and inventory planning for these high-volume products. Investigate alternative warehouse arrangements and supply continuity measures where appropriate.

The dataset confirms historical demand concentration but does not establish whether these products can be supplied through alternative warehouses.

## Q5. Which products recorded demand in every year?

In [13]:
con.execute("""
    SELECT
        Product_Code,
        Product_Category,
        COUNT(DISTINCT YEAR(Date)) AS Active_Years,
        SUM(Order_Demand) AS Positive_Demand,
        COUNT(*) AS Positive_Records
    FROM product_demand
    WHERE Order_Demand > 0
    GROUP BY Product_Code, Product_Category
    HAVING COUNT(DISTINCT YEAR(Date)) = 5
    ORDER BY Positive_Demand DESC
    LIMIT 10
""").df()

,Product_Code,Product_Category,Active_Years,Positive_Demand,Positive_Records
0,Product_1359,Category_019,5,471492000.0,16860
1,Product_1248,Category_019,5,289022000.0,6315
2,Product_0083,Category_019,5,210539000.0,1981
3,Product_1341,Category_019,5,166411000.0,2649
4,Product_1295,Category_019,5,122647500.0,10443
5,Product_1241,Category_019,5,116501000.0,5441
6,Product_1245,Category_019,5,102337000.0,567
7,Product_1286,Category_019,5,101449100.0,8790
8,Product_1432,Category_019,5,96928000.0,2925
9,Product_1274,Category_019,5,92783000.0,7021


In [14]:
con.execute("""
    WITH active_products AS (
        SELECT Product_Code
        FROM product_demand
        WHERE Order_Demand > 0
        GROUP BY Product_Code
        HAVING COUNT(DISTINCT YEAR(Date)) = 5
    )
    SELECT
        COUNT(*) AS Consistently_Active_Products,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(DISTINCT Product_Code)
             FROM product_demand), 2
        ) AS Product_Share
    FROM active_products
""").df()

,Consistently_Active_Products,Product_Share
0,1920,88.89


## Q6. Which products consistently experienced annual demand growth?

In [15]:
con.execute("""
    WITH annual_demand AS (
        SELECT
            Product_Code,
            YEAR(Date) AS Year,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        GROUP BY Product_Code, YEAR(Date)
    ),
    growth_analysis AS (
        SELECT *,
            LAG(Total_Demand) OVER (
                PARTITION BY Product_Code
                ORDER BY Year
            ) AS Previous_Year_Demand
        FROM annual_demand
    )
    SELECT
        Product_Code,
        COUNT(*) AS Active_Years,
        SUM(Total_Demand) AS Total_Demand,
        SUM(CASE
            WHEN Total_Demand > Previous_Year_Demand
            THEN 1 ELSE 0
        END) AS Growth_Years
    FROM growth_analysis
    GROUP BY Product_Code
    HAVING COUNT(*) = 5
       AND SUM(CASE
            WHEN Total_Demand > Previous_Year_Demand
            THEN 1 ELSE 0
        END) = 4
    ORDER BY Total_Demand DESC
    LIMIT 10
""").df()

,Product_Code,Active_Years,Total_Demand,Growth_Years
0,Product_1157,5,34600000.0,4.0
1,Product_1502,5,32640250.0,4.0
2,Product_1470,5,20262700.0,4.0
3,Product_1267,5,10164000.0,4.0
4,Product_1375,5,6262000.0,4.0
5,Product_1402,5,3364100.0,4.0
6,Product_0982,5,3302200.0,4.0
7,Product_1637,5,3300000.0,4.0
8,Product_1314,5,3048300.0,4.0
9,Product_1969,5,2808900.0,4.0


## Q7. Which products had positive demand in 2015 but no positive demand in 2016?

In [16]:
con.execute("""
    SELECT
        Product_Code,
        Product_Category,
        SUM(Order_Demand) AS Demand_2015,
        COUNT(*) AS Records_2015
    FROM product_demand p
    WHERE YEAR(Date) = 2015
      AND Order_Demand > 0
      AND NOT EXISTS (
          SELECT 1
          FROM product_demand p2
          WHERE p2.Product_Code = p.Product_Code
            AND YEAR(p2.Date) = 2016
            AND p2.Order_Demand > 0
      )
    GROUP BY Product_Code, Product_Category
    ORDER BY Demand_2015 DESC
    LIMIT 10
""").df()

,Product_Code,Product_Category,Demand_2015,Records_2015
0,Product_0938,Category_006,985000.0,33
1,Product_1358,Category_019,486000.0,34
2,Product_1184,Category_030,112500.0,12
3,Product_1741,Category_030,70800.0,48
4,Product_1073,Category_020,53292.0,56
5,Product_0372,Category_025,51000.0,5
6,Product_1395,Category_019,46000.0,17
7,Product_0869,Category_005,39000.0,23
8,Product_1067,Category_020,31865.0,38
9,Product_0171,Category_021,29903.0,22


## Q8. Which products experienced consecutive monthly demand declines?

In [17]:
con.execute("""
    WITH monthly_demand AS (
        SELECT
            Product_Code,
            DATE_TRUNC('month', Date) AS Month,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        GROUP BY Product_Code, DATE_TRUNC('month', Date)
    ),
    previous_demand AS (
        SELECT *,
            LAG(Month, 1) OVER (
                PARTITION BY Product_Code
                ORDER BY Month
            ) AS Previous_Month,
            LAG(Month, 2) OVER (
                PARTITION BY Product_Code
                ORDER BY Month
            ) AS Second_Previous_Month,
            LAG(Total_Demand, 1) OVER (
                PARTITION BY Product_Code
                ORDER BY Month
            ) AS Previous_Demand,
            LAG(Total_Demand, 2) OVER (
                PARTITION BY Product_Code
                ORDER BY Month
            ) AS Second_Previous_Demand
        FROM monthly_demand
    )
    SELECT
        Product_Code,
        Month,
        Second_Previous_Demand,
        Previous_Demand,
        Total_Demand,
        Second_Previous_Demand - Total_Demand
            AS Demand_Reduction
    FROM previous_demand
    WHERE Total_Demand < Previous_Demand
      AND Previous_Demand < Second_Previous_Demand
      AND DATE_DIFF('month', Previous_Month, Month) = 1
      AND DATE_DIFF('month', Second_Previous_Month, Month) = 2
    ORDER BY Demand_Reduction DESC
    LIMIT 10
""").df()

,Product_Code,Month,Second_Previous_Demand,Previous_Demand,Total_Demand,Demand_Reduction
0,Product_1341,2012-09-01,6879000.0,3583000.0,1758000.0,5121000.0
1,Product_1248,2012-09-01,6775000.0,3442000.0,2489000.0,4286000.0
2,Product_1248,2012-04-01,6477000.0,4533000.0,2557000.0,3920000.0
3,Product_1245,2014-08-01,4828000.0,2171000.0,916000.0,3912000.0
4,Product_1341,2012-04-01,6863000.0,5306000.0,3038000.0,3825000.0
5,Product_0083,2016-12-01,5728000.0,5445000.0,2042000.0,3686000.0
6,Product_1248,2015-02-01,7726000.0,6534000.0,4044000.0,3682000.0
7,Product_1359,2014-12-01,9750000.0,8306000.0,6198000.0,3552000.0
8,Product_1248,2016-08-01,6509000.0,5091000.0,3077000.0,3432000.0
9,Product_0083,2013-03-01,3535000.0,2171000.0,479000.0,3056000.0


### Key Findings

- The ten highest-demand products with demand recorded in a single warehouse contribute approximately 1.499 billion units.
- These products account for 29.41% of overall net demand and 44.82% of Whse_J's net demand.
- All ten products belong to Category_019, indicating significant warehouse and category-level demand concentration.
- Product_1341 experienced the largest consecutive monthly demand reduction of 5.12 million units between July and September 2012.
- Product_1248 appeared four times among the top ten periods of consecutive monthly demand decline.

### Business Recommendations

- Prioritise monitoring high-volume products with concentrated warehouse demand.
- Regularly monitor monthly demand changes to identify sustained reductions.
- Review seasonal patterns before making inventory adjustments based on short-term demand declines.

## Q9. How concentrated is demand within each warehouse's product portfolio?

In [18]:
con.execute("""
    WITH product_totals AS (
        SELECT
            Warehouse,
            Product_Code,
            SUM(Order_Demand) AS Total_Demand
        FROM product_demand
        WHERE Order_Demand > 0
        GROUP BY Warehouse, Product_Code
    ),
    cumulative_demand AS (
        SELECT *,
            SUM(Total_Demand) OVER (
                PARTITION BY Warehouse
                ORDER BY Total_Demand DESC, Product_Code
                ROWS BETWEEN UNBOUNDED PRECEDING
                AND CURRENT ROW
            ) AS Cumulative_Demand,

            SUM(Total_Demand) OVER (
                PARTITION BY Warehouse
            ) AS Warehouse_Demand

        FROM product_totals
    )
    SELECT
        Warehouse,
        COUNT(*) AS Total_Products,

        SUM(CASE
            WHEN Cumulative_Demand - Total_Demand
                 < Warehouse_Demand * 0.80
            THEN 1 ELSE 0
        END) AS Products_Contributing_80_Pct,

        ROUND(
            SUM(CASE
                WHEN Cumulative_Demand - Total_Demand
                     < Warehouse_Demand * 0.80
                THEN 1 ELSE 0
            END) * 100.0 / COUNT(*), 2
        ) AS Product_Percentage

    FROM cumulative_demand
    GROUP BY Warehouse
    ORDER BY Product_Percentage
""").df()

,Warehouse,Total_Products,Products_Contributing_80_Pct,Product_Percentage
0,Whse_J,1625,67.0,4.12
1,Whse_S,553,27.0,4.88
2,Whse_C,244,22.0,9.02
3,Whse_A,427,46.0,10.77


## Q10. Which product categories have the highest negative demand quantity relative to positive demand?

In [19]:
con.execute("""
    WITH category_summary AS (
        SELECT
            Product_Category,

            SUM(CASE WHEN Order_Demand > 0
                THEN Order_Demand ELSE 0
            END) AS Positive_Demand,

            ABS(SUM(CASE WHEN Order_Demand < 0
                THEN Order_Demand ELSE 0
            END)) AS Negative_Demand

        FROM product_demand
        GROUP BY Product_Category
    )
    SELECT *,
        ROUND(
            Negative_Demand * 100.0 /
            NULLIF(Positive_Demand, 0), 2
        ) AS Negative_Demand_Ratio

    FROM category_summary
    WHERE Positive_Demand > 0
    ORDER BY Negative_Demand_Ratio DESC
    LIMIT 10
""").df()

,Product_Category,Positive_Demand,Negative_Demand,Negative_Demand_Ratio
0,Category_027,80.0,23.0,28.75
1,Category_029,23020.0,959.0,4.17
2,Category_017,47883.0,1574.0,3.29
3,Category_010,21836.0,718.0,3.29
4,Category_012,68208.0,1728.0,2.53
5,Category_009,3694371.0,86193.0,2.33
6,Category_028,48054024.0,884504.0,1.84
7,Category_021,4406379.0,65719.0,1.49
8,Category_015,822997.0,11104.0,1.35
9,Category_013,305423.0,3788.0,1.24


## Q11. How frequently do products experience positive demand?

In [20]:
con.execute("""
    WITH product_frequency AS (
        SELECT
            Product_Code,

            COUNT(DISTINCT CASE
                WHEN Order_Demand > 0
                THEN DATE_TRUNC('month', Date)
            END) AS Active_Months,

            SUM(CASE WHEN Order_Demand > 0
                THEN Order_Demand ELSE 0
            END) AS Positive_Demand

        FROM product_demand
        GROUP BY Product_Code
    ),
    frequency_segments AS (
        SELECT *,
            CASE
                WHEN Active_Months >= 49
                    THEN 'Very Frequent'
                WHEN Active_Months >= 25
                    THEN 'Frequent'
                WHEN Active_Months >= 13
                    THEN 'Occasional'
                WHEN Active_Months >= 1
                    THEN 'Infrequent'
                ELSE 'No Positive Demand'
            END AS Demand_Frequency
        FROM product_frequency
    )
    SELECT
        Demand_Frequency,
        COUNT(*) AS Total_Products,
        SUM(Positive_Demand) AS Total_Demand,

        ROUND(
            COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (), 2
        ) AS Product_Share

    FROM frequency_segments
    GROUP BY Demand_Frequency
    ORDER BY MAX(Active_Months) DESC
""").df()

,Demand_Frequency,Total_Products,Total_Demand,Product_Share
0,Very Frequent,1419,5.018843e+09,65.69
1,Frequent,540,8.595323e+07,25.00
2,Occasional,109,4.099426e+06,5.05
3,Infrequent,92,1.885286e+06,4.26


## Q12. Which products experienced declining demand at one warehouse and increasing demand at another?

In [21]:
con.execute("""
    WITH warehouse_product_demand AS (
        SELECT
            Product_Code,
            Warehouse,

            SUM(CASE WHEN YEAR(Date) = 2015
                THEN Order_Demand ELSE 0
            END) AS Demand_2015,

            SUM(CASE WHEN YEAR(Date) = 2016
                THEN Order_Demand ELSE 0
            END) AS Demand_2016

        FROM product_demand
        WHERE YEAR(Date) IN (2015, 2016)
        GROUP BY Product_Code, Warehouse
    ),
    demand_changes AS (
        SELECT *,
            Demand_2016 - Demand_2015 AS Demand_Change
        FROM warehouse_product_demand
    )
    SELECT
        a.Product_Code,
        a.Warehouse AS Declining_Warehouse,
        a.Demand_Change AS Demand_Decline,

        b.Warehouse AS Growing_Warehouse,
        b.Demand_Change AS Demand_Increase

    FROM demand_changes a
    JOIN demand_changes b
        ON a.Product_Code = b.Product_Code
        AND a.Warehouse != b.Warehouse

    WHERE a.Demand_Change < 0
      AND b.Demand_Change > 0

    ORDER BY a.Demand_Change ASC,
             b.Demand_Change DESC

    LIMIT 15
""").df()

,Product_Code,Declining_Warehouse,Demand_Decline,Growing_Warehouse,Demand_Increase
0,Product_1393,Whse_S,-6359000.0,Whse_C,297000.0
1,Product_1393,Whse_S,-6359000.0,Whse_J,57000.0
2,Product_1431,Whse_S,-4044000.0,Whse_C,561000.0
3,Product_1241,Whse_S,-3991000.0,Whse_J,431000.0
4,Product_1264,Whse_J,-2756000.0,Whse_A,125000.0
5,Product_1394,Whse_S,-2599000.0,Whse_C,1570000.0
6,Product_1904,Whse_S,-1912000.0,Whse_J,27000.0
7,Product_1281,Whse_C,-1777000.0,Whse_J,82000.0
8,Product_1281,Whse_S,-1745000.0,Whse_J,82000.0
9,Product_1016,Whse_J,-1671600.0,Whse_C,70600.0


### Key Findings from SQL Business Analysis

- Only 4.12% of products at Whse_J contribute approximately 80% of its positive demand, indicating significant demand concentration.
- Whse_S also demonstrates high demand concentration, with 4.88% of its products contributing approximately 80% of positive demand.
- Category_027 has the highest negative demand ratio of 28.75%, although its absolute negative demand quantity is relatively small.
- Category_028 records a substantial negative demand quantity of 884,504 units despite having a relatively low negative demand ratio of 1.84%.
- A total of 1,419 products are classified as Very Frequent, accounting for 65.69% of all products and approximately 98.20% of positive demand.
- Several products experienced declining demand at one warehouse while experiencing increasing demand at another.
- Product_1394 recorded a demand reduction of 2.599 million units at Whse_S and an increase of 1.570 million units at Whse_C.

### Business Recommendations

- Prioritise high-volume products within individual warehouses.
- Monitor both the frequency and quantity of product demand.
- Evaluate absolute negative demand alongside percentage ratios.
- Review warehouse-specific demand changes before making inventory allocation decisions.
- Avoid applying identical inventory management strategies to products with different demand characteristics.